# Getting started

`seqout` gets sequencing dataset metadata from seqout.org. This notebook
covers the first steps: connect, search, and open a dataset.

In [1]:
from seqout import connect

sq = connect()

## Search

Give the query as a string. Filters go in as keyword arguments.

In [2]:
results = sq.search("pancreatic cancer single cell", db="geo")
for r in results[:5]:
    print(r.accession, "-", r.title)

GSE165399 - Single-cell transcriptomics of normal pancreas, intraductal papillary mucinous neoplasm, and pancreatic adenosquamous carcinoma reveals the heterogeneous progression of pancreatic ductal and stromal cells
GSE274617 - Single cell transcriptomes of pancreatic pre-invasive lesions and cancer reveal acinar metaplastic cells’ heterogeneity [MERFISH]
GSE292513 - Loss of NF2 drives malignant transformation of human pancreatic acinar cells and enhances cell fitness under nutrient deprivation and therapeutical stress [single_cell_RNAseq]
GSE141017 - Single cell transcriptomes of pancreatic pre-invasive lesions and cancer reveal acinar metaplastic cells’ heterogeneity
GSE180859 - Single cell transcriptome analysis (scRNASeq) and inferred single cell copy number variations (scCNVs) of cancer associated fibroblast (CAFs) populations in murine KPC pancreatic tumors


## Open a dataset

`get` accepts any accession — a series, a study, an experiment, a sample, or a
run. It finds the related records for you, so you do not have to know which
archive holds which part.

In [3]:
d = sq.get("GSE169470")

print(d.meta.title)
print("organisms:", d.meta.organisms, "| pmid:", d.meta.pmid)
print("archives: geo =", d.geo, "| sra =", d.sra)

RNA-sequencing Analysis (RNA-seq)，Genome-wide Maps of Chromatin State (ChIP-seq) and Assay for Transposase Accessible Chromatin with High-throughput Sequencing (ATAC-seq) in BMDMs or Raw 264.7 cell lines.
organisms: ['Mus musculus'] | pmid: 34764296


archives: geo = GSE169470 | sra = SRP311850


## Samples and runs

A GEO series holds no sequencing runs; the linked SRA study does. `runs`
crosses that link on its own.

In [4]:
print(len(d.samples), "samples")
print(len(d.runs), "runs from", d.sra)

47 samples


47 runs from SRP311850


Each field makes its request at the first use and keeps the result, so the
lines above cost one request each, not one for every use.

## Run data formats

A run can offer the data in several formats — FASTQ, SRA, and the NCBI cloud
mirrors (NCBI, S3, GCS). Not every format is present for every run. Print the
ones this run has:

In [5]:
run = d.runs[0]
print(run.run_accession)
for fmt, url in {
    "fastq": run.fastq_ftp,
    "sra": run.sra_ftp,
    "ncbi": run.ncbi_sra_url,
    "sra-lite": run.ncbi_sra_lite_url,
    "s3": run.ncbi_sra_lite_s3_url,
    "gcs": run.ncbi_sra_lite_gs_url,
}.items():
    if url:
        print(f"  {fmt:9} {url}")

SRR14049273
  ncbi      https://sra-downloadb.be-md.ncbi.nlm.nih.gov/sos4/sra-pub-zq-1/SRR014/14049/SRR14049273/SRR14049273.lite.1
  sra-lite  https://sra-downloadb.be-md.ncbi.nlm.nih.gov/sos4/sra-pub-zq-1/SRR014/14049/SRR14049273/SRR14049273.lite.1
  s3        s3://sra-pub-zq-6/SRR14049273/SRR14049273.lite.1
  gcs       gs://sra-pub-zq-104/SRR14049273/SRR14049273.lite.1


## Start from any accession

A sample or a run resolves to its parent in the same way.

In [6]:
sq.get(run.run_accession).project

'SRP311850'